In [12]:
pip install duckdb
pip install httpx
pip install sqlonfhir

Note: you may need to restart the kernel to use updated packages.


  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [httpx]
Note: you may need to restart the kernel to use updated packages.


  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sqlonfhir]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.13.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import duckdb
import httpx
import pandas as pd
from sqlonfhir import evaluate

In [13]:
BASE = "https://demo.kodjin.com/fhir"
PAGE_SIZE = 50
MAX_PATIENTS = 100

In [21]:
def fetch_bundle(
    client: httpx.Client, url: str, params: dict | None = None, max_pages: int = 10
) -> list[dict]:
    resources = []
    next_url, next_params = url, params
    pages = 0
    while next_url and pages < max_pages:
        r = client.get(next_url, params=next_params)
        r.raise_for_status()
        bundle = r.json()
        resources.extend(
            e["resource"] for e in bundle.get("entry", []) if "resource" in e
        )
        next_url = next(
            (l["url"] for l in bundle.get("link", []) if l["relation"] == "next"), None
        )
        next_params = None
        pages += 1
    return resources


In [22]:
VITAL_LOINCS = [
    "8867-4",  # Heart rate
    "8480-6",  # Systolic BP
    "8462-4",  # Diastolic BP
    "29463-7",  # Body weight
    "8302-2",  # Body height
    "39156-5",  # BMI
    "8310-5",  # Body temperature
    "9279-1",  # Respiratory rate
    "4548-4",  # HbA1c
]

In [23]:
with httpx.Client(timeout=30.0, headers={"Accept": "application/fhir+json"}) as client:
    # Start from observations with known vital-sign LOINCs
    code_param = ",".join(f"http://loinc.org|{c}" for c in VITAL_LOINCS)
    observations = fetch_bundle(
        client,
        f"{BASE}/Observation",
        {
            "code": code_param,
            "_count": PAGE_SIZE,
        },
        max_pages=2,
    )

    # Collect referenced patient IDs
    patient_ids = sorted(
        {
            ref.removeprefix("Patient/")
            for o in observations
            if (ref := o.get("subject", {}).get("reference", "")).startswith("Patient/")
        }
    )

    # Fetch those patients in batches
    patients = []
    for i in range(0, len(patient_ids), 20):
        chunk = ",".join(patient_ids[i : i + 20])
        patients.extend(
            fetch_bundle(client, f"{BASE}/Patient", {"_id": chunk, "_count": PAGE_SIZE})
        )

    print(f"Fetched {len(patients)} patients, {len(observations)} observations")

Fetched 0 patients, 0 observations


In [17]:
# ViewDefinitions
patient_view = {
    "resource": "Patient",
    "select": [
        {
            "column": [
                {"name": "id", "path": "id"},
                {"name": "gender", "path": "gender"},
                {"name": "birth_date", "path": "birthDate"},
            ]
        },
        {
            "forEachOrNull": "name.where(use = 'official').first() | name.first()",
            "column": [
                {"name": "family", "path": "family"},
                {"name": "given", "path": "given.first()"},
            ],
        },
    ],
}


In [18]:
observation_view = {
    "resource": "Observation",
    "select": [
        {
            "column": [
                {"name": "id", "path": "id"},
                {
                    "name": "patient_id",
                    "path": "subject.reference.replace('Patient/', '')",
                },
                {"name": "status", "path": "status"},
                {"name": "effective", "path": "effective.ofType(dateTime)"},
                {"name": "value_quantity", "path": "value.ofType(Quantity).value"},
                {"name": "value_unit", "path": "value.ofType(Quantity).unit"},
            ]
        },
        {
            "forEachOrNull": "code.coding.where(system = 'http://loinc.org').first() | code.coding.first()",
            "column": [
                {"name": "code_system", "path": "system"},
                {"name": "code", "path": "code"},
                {"name": "display", "path": "display"},
            ],
        },
    ],
}

In [19]:
# Flatten and load into DuckDB
con = duckdb.connect()
con.register("patient_demo", pd.DataFrame(evaluate(patients, patient_view)))
con.register("observation_flat", pd.DataFrame(evaluate(observations, observation_view)))

InvalidInputException: Invalid Input Error: Need a DataFrame with at least one column

In [ ]:
print("\n--- Patients ---")
con.sql("SELECT gender, COUNT(*) AS n FROM patient_demo GROUP BY gender").show()

In [ ]:
print("\n--- Top observation codes ---")
con.sql("""
    SELECT code, display, COUNT(*) AS n
    FROM observation_flat
    WHERE code IS NOT NULL
    GROUP BY code, display
    ORDER BY n DESC
    LIMIT 10
""").show()

In [ ]:
print("\n--- Numeric obs diagnostics ---")
con.sql("""
    SELECT
      COUNT(*) AS total_obs,
      COUNT(value_quantity) AS with_numeric_value,
      COUNT(DISTINCT code) AS distinct_codes,
      COUNT(DISTINCT patient_id) AS distinct_patients
    FROM observation_flat
""").show()

In [ ]:
con.sql("""
    SELECT code, display, value_unit, COUNT(*) AS n
    FROM observation_flat
    WHERE value_quantity IS NOT NULL
    GROUP BY code, display, value_unit
    ORDER BY n DESC
    LIMIT 10
""").show()

In [ ]:
print("\n--- Numeric observations by patient gender ---")
con.sql("""
    SELECT p.gender,
           o.code,
           o.display,
           COUNT(*) AS n,
           ROUND(AVG(o.value_quantity), 2) AS mean_value,
           o.value_unit
    FROM observation_flat o
    JOIN patient_demo p ON p.id = o.patient_id
    WHERE o.value_quantity IS NOT NULL
    GROUP BY p.gender, o.code, o.display, o.value_unit
    HAVING COUNT(*) >= 3
    ORDER BY n DESC
    LIMIT 15
""").show()